# 0. Problem
## 626. Exchange Seats — Medium
Swap each adjacent pair of seat IDs. If the last seat is unpaired, leave it unchanged.
Official: https://leetcode.com/problems/exchange-seats/

# 1. Setup

In [ ]:
import pandas as pd
import numpy as np
seat_rows=[(1,"Abbot"),(2,"Doris"),(3,"Emerson"),(4,"Green"),(5,"Jeames")]
seat_pd=pd.DataFrame(seat_rows,columns=["id","student"])
seat_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
seat_spark=spark.createDataFrame(seat_rows,["id","student"])
seat_spark.createOrReplaceTempView("Seat")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""SELECT CASE WHEN id%2=1 AND id=(SELECT MAX(id) FROM Seat) THEN id WHEN id%2=1 THEN id+1 ELSE id-1 END AS id,student FROM Seat ORDER BY id""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
max_id=seat_pd["id"].max()
result_pd=seat_pd.copy()
result_pd["id"]=np.select([result_pd["id"].mod(2).eq(1)&result_pd["id"].eq(max_id),result_pd["id"].mod(2).eq(1)],[result_pd["id"],result_pd["id"]+1],default=result_pd["id"]-1)
result_pd=result_pd.sort_values("id").reset_index(drop=True)
result_pd

# 4. PySpark Solution

In [ ]:
max_id=seat_spark.agg(F.max("id").alias("max_id")).first()["max_id"]
result_spark=(seat_spark.withColumn("id",F.when((F.col("id")%2==1)&(F.col("id")==max_id),F.col("id")).when(F.col("id")%2==1,F.col("id")+1).otherwise(F.col("id")-1)).orderBy("id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| conditional remap | `CASE WHEN` | `np.select()` | chained `F.when()` |
| scalar max | subquery `MAX()` | `.max()` | `.agg(F.max())` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Seat

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: seat_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: seat_spark